# Tokenization Deep Dive — Part 2: HuggingFace & Training a Small Model

---

| # | Topic |
|---|-------|
| 7 | **HuggingFace `tokenizers` Library** — fast tokenizer training |
| 8 | **Pre-trained Tokenizers** — GPT-2 vs BERT side by side |
| 9 | **Train a Custom BPE Tokenizer** — on a real dataset |
| 10 | **Train a Small GPT-style Language Model** — end-to-end |
| 11 | **Key Takeaways** |

In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import AutoTokenizer

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
# Part 7 — HuggingFace `tokenizers` Library

The `tokenizers` library (written in Rust) provides blazing-fast implementations of all major algorithms.

### Key components
- **Model**: The core algorithm (BPE, WordPiece, Unigram)
- **Pre-tokenizer**: How to split input before applying the model (whitespace, punctuation, byte-level)
- **Trainer**: How to learn the vocabulary from data
- **Decoder**: How to reconstruct text from tokens

In [ ]:
# ---- Train a BPE tokenizer using HuggingFace tokenizers ----

# Step 1: Initialize a BPE model
tokenizer = Tokenizer(models.BPE())

# Step 2: Set a pre-tokenizer (split on whitespace + punctuation)
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# Step 3: Configure trainer
trainer = trainers.BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"],
    show_progress=True
)

# Step 4: Prepare training data (we'll use our sample corpus inline)
corpus = [
    "Tokenization is the first step in any NLP pipeline.",
    "Tokens can be words, subwords, or characters.",
    "Subword tokenization balances vocabulary size and sequence length.",
    "BPE and WordPiece are the most popular subword methods.",
    "GPT uses BPE tokenization while BERT uses WordPiece.",
    "The tokenizer converts text into token IDs that the model understands.",
    "A good tokenizer should handle unseen words gracefully.",
    "Training a tokenizer means learning the best way to split text.",
    "Low frequency words get split into smaller subword units.",
    "High frequency words remain as single tokens.",
    "Language models predict the next token in a sequence.",
    "Transformers use self-attention to process all tokens in parallel.",
    "The embedding layer converts token IDs into dense vectors.",
    "Positional encoding tells the model the order of tokens.",
    "Multi-head attention allows the model to focus on different parts.",
    "The feed-forward network processes each position independently.",
    "Layer normalization stabilizes training of deep networks.",
    "Dropout prevents overfitting during training.",
    "The output layer projects hidden states to vocabulary logits.",
    "Cross-entropy loss measures how well the model predicts tokens.",
]

# Write corpus to a temp file (tokenizers library reads from files)
with open("train_corpus.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

# Step 5: Train!
tokenizer.train(["train_corpus.txt"], trainer)

print(f"Vocabulary size: {tokenizer.get_vocab_size()}")
print(f"\nSample vocab entries:")
vocab = tokenizer.get_vocab()
for token, idx in sorted(vocab.items(), key=lambda x: x[1])[:30]:
    print(f"  {idx:3d}: '{token}'")

In [ ]:
# Test the HuggingFace tokenizer
test_texts = [
    "Tokenization is important for NLP.",
    "Transformers process tokens in parallel.",
    "Supercalifragilistic is an unseen word.",
]

for text in test_texts:
    output = tokenizer.encode(text)
    print(f"Input  : {text}")
    print(f"Tokens : {output.tokens}")
    print(f"IDs    : {output.ids}")
    print(f"Offsets: {output.offsets}")
    print(f"Decoded: {tokenizer.decode(output.ids)}")
    print()

---
# Part 8 — Pre-trained Tokenizers: GPT-2 vs BERT

Let's compare how real production tokenizers handle the same text.

In [ ]:
# Load pre-trained tokenizers
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"GPT-2 vocab size : {gpt2_tok.vocab_size}")
print(f"BERT vocab size  : {bert_tok.vocab_size}")

In [ ]:
# Side-by-side comparison
comparison_texts = [
    "Hello, how are you?",
    "Tokenization is fundamental to NLP.",
    "The unhappiness of the unfriendly cat.",
    "GPT-4 and BERT handle tokenization differently.",
    "I love playing football on Sundays!",
    "Pneumonoultramicroscopicsilicovolcanoconiosis",  # longest English word
]

print(f"{'Text':<50} | {'GPT-2 Tokens':>12} | {'BERT Tokens':>12}")
print("-" * 80)

for text in comparison_texts:
    gpt2_tokens = gpt2_tok.tokenize(text)
    bert_tokens = bert_tok.tokenize(text)
    print(f"{text:<50} | {len(gpt2_tokens):>12} | {len(bert_tokens):>12}")

print("\n--- Detailed token comparison ---")
for text in comparison_texts:
    gpt2_tokens = gpt2_tok.tokenize(text)
    bert_tokens = bert_tok.tokenize(text)
    gpt2_ids = gpt2_tok.encode(text)
    bert_ids = bert_tok.encode(text)
    
    print(f"\nText: {text}")
    print(f"  GPT-2  tokens: {gpt2_tokens}")
    print(f"  GPT-2  IDs   : {gpt2_ids}")
    print(f"  BERT   tokens: {bert_tokens}")
    print(f"  BERT   IDs   : {bert_ids}")

In [ ]:
# Special tokens and features comparison
text = "Hello world!"

print("=== GPT-2 (BPE, no special tokens added by default) ===")
gpt2_enc = gpt2_tok(text, return_tensors="pt")
print(f"  input_ids     : {gpt2_enc['input_ids'].tolist()}")
print(f"  attention_mask: {gpt2_enc['attention_mask'].tolist()}")
print(f"  tokens        : {gpt2_tok.convert_ids_to_tokens(gpt2_enc['input_ids'][0])}")
print(f"  decoded       : {gpt2_tok.decode(gpt2_enc['input_ids'][0])}")

print(f"\n=== BERT (WordPiece, adds [CLS] and [SEP]) ===")
bert_enc = bert_tok(text, return_tensors="pt")
print(f"  input_ids     : {bert_enc['input_ids'].tolist()}")
print(f"  attention_mask: {bert_enc['attention_mask'].tolist()}")
print(f"  token_type_ids: {bert_enc['token_type_ids'].tolist()}")
print(f"  tokens        : {bert_tok.convert_ids_to_tokens(bert_enc['input_ids'][0])}")
print(f"  decoded       : {bert_tok.decode(bert_enc['input_ids'][0])}")

print(f"\n=== Key Differences ===")
print("  GPT-2: Byte-level BPE, uses 'G' prefix for space, no [CLS]/[SEP]")
print("  BERT:  WordPiece, uses ## for continuation, adds [CLS] and [SEP]")
print("  GPT-2: Case-sensitive | BERT (uncased): lowercases everything")

---
# Part 9 — Train a Custom BPE Tokenizer on a Real Dataset

Let's train a proper tokenizer using the HuggingFace `datasets` library to fetch real data.

In [ ]:
from datasets import load_dataset

# Load a small subset of wikitext
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
print(f"Dataset size: {len(dataset)} lines")
print(f"\nSample lines:")
for i in range(5):
    if dataset[i]["text"].strip():
        print(f"  {dataset[i]['text'][:120]}")

In [ ]:
# Write dataset to file for tokenizer training
with open("wikitext_train.txt", "w", encoding="utf-8") as f:
    for example in dataset:
        text = example["text"].strip()
        if text:
            f.write(text + "\n")

# Train a BPE tokenizer with a realistic vocabulary size
custom_tokenizer = Tokenizer(models.BPE(unk_token="<UNK>"))
custom_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

custom_trainer = trainers.BpeTrainer(
    vocab_size=8000,
    min_frequency=2,
    special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>"],
    show_progress=True
)

custom_tokenizer.train(["wikitext_train.txt"], custom_trainer)
custom_tokenizer.decoder = decoders.ByteLevel()

# Save the tokenizer
custom_tokenizer.save("custom_bpe_tokenizer.json")

print(f"\nCustom tokenizer vocab size: {custom_tokenizer.get_vocab_size()}")

# Test it
test = "Transformers have revolutionized natural language processing."
enc = custom_tokenizer.encode(test)
print(f"\nTest: {test}")
print(f"Tokens ({len(enc.tokens)}): {enc.tokens}")
print(f"IDs: {enc.ids}")
print(f"Decoded: {custom_tokenizer.decode(enc.ids)}")

In [ ]:
# Visualize token length distribution
sample_texts = [ex["text"] for ex in dataset if ex["text"].strip()][:500]

custom_lengths = []
for text in sample_texts:
    enc = custom_tokenizer.encode(text)
    custom_lengths.append(len(enc.ids))

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
ax.hist(custom_lengths, bins=50, alpha=0.7, color="steelblue", edgecolor="black")
ax.set_xlabel("Number of tokens")
ax.set_ylabel("Frequency")
ax.set_title("Token count distribution (Custom BPE, vocab=8000)")
ax.axvline(np.mean(custom_lengths), color="red", linestyle="--", label=f"Mean: {np.mean(custom_lengths):.1f}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean tokens per line: {np.mean(custom_lengths):.1f}")
print(f"Max tokens per line : {np.max(custom_lengths)}")

---
# Part 10 — Training a Small GPT-style Language Model

Now we'll bring everything together and train a **small causal language model** (GPT-style) from scratch using our custom tokenizer.

### Architecture
- **Embedding layer**: token IDs -> dense vectors
- **Positional encoding**: learned positional embeddings
- **Transformer decoder blocks**: masked self-attention + feed-forward
- **Output head**: project back to vocabulary size

### Hyperparameters (small, trainable on CPU)
- `d_model = 128`, `n_heads = 4`, `n_layers = 4`, `block_size = 64`

In [ ]:
# ============================================================
# Model Definition: A Small GPT
# ============================================================

class MultiHeadSelfAttention(nn.Module):
    """Multi-head masked self-attention."""
    
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
        
        # Causal mask
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size)).unsqueeze(0).unsqueeze(0)
        )
    
    def forward(self, x):
        B, T, C = x.shape
        
        # Compute Q, K, V
        qkv = self.qkv(x)  # (B, T, 3*C)
        q, k, v = qkv.chunk(3, dim=-1)  # Each: (B, T, C)
        
        # Reshape to (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        attn = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        
        # Weighted sum
        out = attn @ v  # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj_dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dropout)
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # Pre-norm residual
        x = x + self.ff(self.ln2(x))
        return x


class SmallGPT(nn.Module):
    """
    A small GPT-style causal language model.
    """
    
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, block_size=64, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        
        self.blocks = nn.Sequential(
            *[TransformerBlock(d_model, n_heads, block_size, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying
        self.head.weight = self.token_emb.weight
        
        self.apply(self._init_weights)
        
        n_params = sum(p.numel() for p in self.parameters())
        print(f"SmallGPT: {n_params:,} parameters")
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, f"Sequence length {T} > block size {self.block_size}"
        
        tok_emb = self.token_emb(idx)          # (B, T, d_model)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, d_model)
        x = self.drop(tok_emb + pos_emb)
        
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)                  # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Autoregressive generation."""
        for _ in range(max_new_tokens):
            # Crop to block_size
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


print("Model class defined!")

In [ ]:
# ============================================================
# Dataset: Prepare training data using our custom tokenizer
# ============================================================

BLOCK_SIZE = 64  # context window

class TextDataset(Dataset):
    """Simple dataset that creates overlapping chunks of token IDs."""
    
    def __init__(self, token_ids, block_size):
        self.block_size = block_size
        self.data = torch.tensor(token_ids, dtype=torch.long)
    
    def __len__(self):
        return max(0, len(self.data) - self.block_size - 1)
    
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y


# Load our custom tokenizer
tok = Tokenizer.from_file("custom_bpe_tokenizer.json")
VOCAB_SIZE = tok.get_vocab_size()

# Tokenize the entire wikitext training set
print("Tokenizing dataset...")
all_ids = []
for example in tqdm(dataset):
    text = example["text"].strip()
    if text:
        encoded = tok.encode(text)
        all_ids.extend(encoded.ids)

print(f"Total tokens: {len(all_ids):,}")
print(f"Vocab size  : {VOCAB_SIZE}")

# Create dataset and dataloader
train_dataset = TextDataset(all_ids, BLOCK_SIZE)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True)

print(f"Training examples: {len(train_dataset):,}")
print(f"Batches per epoch: {len(train_loader):,}")

# Peek at a sample
x, y = train_dataset[0]
print(f"\nSample input  (x): {x[:10].tolist()} ...")
print(f"Sample target (y): {y[:10].tolist()} ...")
print(f"Decoded x: {tok.decode(x.tolist())[:80]}...")

In [ ]:
# ============================================================
# Training Loop
# ============================================================

# Hyperparameters
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
LEARNING_RATE = 3e-4
NUM_EPOCHS = 3
EVAL_INTERVAL = 200  # steps between logging

# Create model
model = SmallGPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    block_size=BLOCK_SIZE,
    dropout=0.1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Training
losses = []
model.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    
    for step, (x, y) in enumerate(pbar):
        x, y = x.to(device), y.to(device)
        
        logits, loss = model(x, targets=y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        losses.append(loss.item())
        epoch_loss += loss.item()
        
        if (step + 1) % EVAL_INTERVAL == 0:
            avg_loss = sum(losses[-EVAL_INTERVAL:]) / EVAL_INTERVAL
            pbar.set_postfix({"loss": f"{avg_loss:.4f}"})
    
    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch + 1} avg loss: {avg_epoch_loss:.4f}")

In [ ]:
# Plot training loss
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Raw loss
axes[0].plot(losses, alpha=0.3, color="steelblue")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss (per step)")

# Smoothed loss
window = 100
if len(losses) > window:
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    axes[1].plot(smoothed, color="darkorange")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss (smoothed)")
    axes[1].set_title(f"Training Loss (smoothed, window={window})")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Text Generation
# ============================================================

model.eval()

prompts = [
    "The history of",
    "In the beginning",
    "Language models are",
    "The president",
]

for prompt in prompts:
    # Encode prompt
    prompt_ids = tok.encode(prompt).ids
    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    
    # Generate
    output_ids = model.generate(input_ids, max_new_tokens=50, temperature=0.8, top_k=40)
    generated_text = tok.decode(output_ids[0].tolist())
    
    print(f"Prompt: {prompt}")
    print(f"Generated: {generated_text}")
    print("-" * 80)

In [ ]:
# ============================================================
# Inspect: What did the tokenizer and model learn?
# ============================================================

# 1. Most common tokens in our vocabulary
from collections import Counter

token_counts = Counter(all_ids)
print("Top 20 most frequent tokens in the training data:")
print("-" * 40)
for token_id, count in token_counts.most_common(20):
    token_str = tok.decode([token_id])
    print(f"  ID {token_id:5d} | count {count:6d} | '{token_str}'")

# 2. Token embedding similarity
print("\n\nToken Embedding Cosine Similarities:")
print("-" * 40)

def get_token_embedding(text):
    ids = tok.encode(text).ids
    if len(ids) == 0:
        return None
    emb = model.token_emb.weight[ids[0]]
    return emb / emb.norm()

word_pairs = [("the", "a"), ("is", "was"), ("the", "and"), ("he", "she")]
for w1, w2 in word_pairs:
    e1 = get_token_embedding(w1)
    e2 = get_token_embedding(w2)
    if e1 is not None and e2 is not None:
        sim = (e1 @ e2).item()
        print(f"  cos('{w1}', '{w2}') = {sim:.4f}")

---
# Part 11 — Key Takeaways

### Tokenization Strategy
- **Character-level**: Simple, no OOV, but sequences are too long
- **Word-level**: Semantically rich tokens, but huge vocab and OOV problems
- **Subword (BPE/WordPiece)**: Best of both worlds — manageable vocab, short sequences, rare OOV

### BPE vs WordPiece
- **BPE**: Merges most frequent pairs (used by GPT, LLaMA)
- **WordPiece**: Merges by mutual information score (used by BERT)
- In practice, both produce very similar results

### Practical Tips
1. **Vocab size matters**: Too small = long sequences; too large = sparse embeddings
2. **Always use the same tokenizer** for training and inference
3. **Special tokens** ([CLS], [SEP], <BOS>, <EOS>) serve structural roles
4. **Byte-level BPE** (GPT-2 style) can handle any UTF-8 input without UNK tokens
5. **Tokenizer quality directly impacts model quality** — garbage tokenization = garbage model

### What We Built
1. Character, Word, BPE, and WordPiece tokenizers **from scratch**
2. Trained a BPE tokenizer on WikiText-2 using HuggingFace `tokenizers`
3. Trained a **~1M parameter GPT-style model** end-to-end
4. Generated text with our custom model + tokenizer

### Further Reading
- [HuggingFace Tokenizers docs](https://huggingface.co/docs/tokenizers)
- [HuggingFace NLP Course - Tokenizers](https://huggingface.co/learn/nlp-course/chapter6)
- [Karpathy: Let's build GPT from scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY)
- [Sennrich et al., 2016 — Original BPE paper](https://arxiv.org/abs/1508.07909)
- [SentencePiece — Unigram LM tokenization](https://arxiv.org/abs/1808.06226)